# Lección 4 · Asignación

**Qué vas a aprender**
- El **problema de asignación**: quién hace qué, al menor costo.
- Por qué la regla "primero el par más barato" falla.
- Por qué aquí el LP **ya da 0 y 1** sin branch & bound: la **matriz totalmente unimodular (TU)**.
- Qué restricción extra rompe esa suerte.
- **Pares factibles**: crear solo las variables posibles con un *join*, como en SQL.
- Revisar los datos **antes** de resolver, y un modelo **elástico** que no se cae si falta gente.
- 50 ingenieros × 20 proyectos, con la solución en tablas y los precios sombra de vuelta.

**Cómo usar este notebook**
- Corre las celdas en orden con `Shift + Enter`.
- Antes de cada celda, adivina qué va a imprimir.

## 0. Preparar el entorno
Igual que en las lecciones anteriores. `scipy` ya viene en Colab: la usamos en la sección 9.

In [ ]:
%pip install -q pyomo highspy

In [ ]:
import itertools
import io
import time

import numpy as np
import pandas as pd
import pyomo.environ as pyo
from scipy.optimize import linear_sum_assignment

SOLVER = pyo.SolverFactory("appsi_highs")


def eur(x):
    return f"{x:,.0f}".replace(",", ".") + " €"

## 1. Dónde quedamos
En la lección 3 las binarias obligaron al solver a hacer **branch & bound**: partir, acotar, podar.
Con modelos grandes eso puede tardar horas.

Hoy ves un modelo con binarias donde el solver **no necesita ramificar**: el LP ya sale con 0 y 1.
Es el problema de **asignación**, uno de los más comunes en la industria:
ingenieros a proyectos, asesores a clientes, camiones a rutas, turnos a personas.

**Analogía.** El sábado hay tres tareas: llevar a tu hijo al parque, hacer el mercado y cocinar.
Están tú, tu esposa y la babcia. Cada uno hace una tarea, y cada tarea la hace uno.
A cada uno le cuesta distinto: la babcia cocina rápido; tú tardas en el mercado porque lees las etiquetas en polaco.
¿Quién hace qué para que el sábado cueste menos?

## 2. Tres ingenieros, tres proyectos
La empresa de la lección 3 tiene que poner un ingeniero a cargo de cada proyecto.
La tabla dice cuánto cuesta cada ingeniero en cada proyecto (miles de €: tarifa, viajes, curva de aprendizaje).

| | solar | eólico | subestación |
|---|---|---|---|
| **Ana** | 4 | 7 | 3 |
| **Bartek** | 2 | 6 | 5 |
| **Camilo** | 5 | 10 | 6 |

**Las 3 piezas:**
- **Decido:** x[i, j] = 1 si el ingeniero i va al proyecto j, 0 si no. Son 3 × 3 = 9 binarias.
- **Objetivo:** minimizar Σ costo[i, j] · x[i, j].
- **Restricciones:**
  - cada ingeniero va a un proyecto: Σ_j x[i, j] = 1, para cada i;
  - cada proyecto tiene un ingeniero: Σ_i x[i, j] = 1, para cada j.

In [ ]:
COSTO = pd.DataFrame([[4, 7, 3],
                      [2, 6, 5],
                      [5, 10, 6]],
                     index=["Ana", "Bartek", "Camilo"], columns=["solar", "eolico", "subestacion"])
COSTO

### 2.1 Fuerza bruta
Una asignación es un orden de los proyectos: Ana toma el primero, Bartek el segundo, Camilo el tercero.
Hay 3 × 2 × 1 = 3! = **6** órdenes posibles. `itertools.permutations` los genera todos.

In [ ]:
filas = []
for orden in itertools.permutations(COSTO.columns):
    plan = dict(zip(COSTO.index, orden))
    filas.append({**plan, "costo": sum(COSTO.loc[i, j] for i, j in plan.items())})
pd.DataFrame(filas).sort_values("costo")

**Lectura:** el mejor plan es **Ana → subestación, Bartek → eólico, Camilo → solar**: 14 mil €.

Con 10 ingenieros son 10! ≈ 3,6 millones de órdenes. Con 50, un número de 65 cifras. Fuerza bruta, otra vez, no.

### 2.2 El atajo tentador: primero el par más barato
Un jefe de proyectos haría esto: busca la celda más barata de la tabla, asigna ese par, tacha su fila y su columna,
y repite. Se llama **algoritmo goloso** (*greedy*): en cada paso toma lo mejor que ve, sin mirar adelante.

In [ ]:
def goloso(costo):
    libres_i, libres_j, plan = set(costo.index), set(costo.columns), {}
    while libres_i and libres_j:
        i, j = min(((i, j) for i in libres_i for j in libres_j), key=lambda par: costo.loc[par])
        plan[i] = j
        libres_i.remove(i)
        libres_j.remove(j)
        print(f"  {i:<7} → {j:<12} {costo.loc[i, j]}")
    return plan


plan_goloso = goloso(COSTO)
print("Costo total:", sum(COSTO.loc[i, j] for i, j in plan_goloso.items()))

**Lectura:** el goloso toma Bartek → solar (2, la celda más barata) y Ana → subestación (3).
A Camilo le queda solo el eólico, el peor par de la tabla (10). Total **15**, contra 14 del óptimo.

El óptimo **no usa** la celda más barata. Lo mismo que en la lección 3: no importa cuánto vale cada pieza,
importa **cómo encajan**. El par Bartek → solar es barato, pero obliga a Camilo a ir al eólico.

## 3. Asignación en Pyomo
Las dos restricciones son una por ingeniero y una por proyecto. El parámetro `relajar` es el de la lección 3:
`pyo.Binary` (solo 0 o 1) o `pyo.UnitInterval` (cualquier número entre 0 y 1).

In [ ]:
def asignar(costo=COSTO, relajar=False):
    m = pyo.ConcreteModel()
    m.I = pyo.Set(initialize=list(costo.index))    # ingenieros
    m.J = pyo.Set(initialize=list(costo.columns))  # proyectos
    m.x = pyo.Var(m.I, m.J, within=pyo.UnitInterval if relajar else pyo.Binary)
    m.obj = pyo.Objective(expr=sum(costo.loc[i, j] * m.x[i, j] for i in m.I for j in m.J), sense=pyo.minimize)
    m.ingeniero = pyo.Constraint(m.I, rule=lambda m, i: sum(m.x[i, j] for j in m.J) == 1)
    m.proyecto = pyo.Constraint(m.J, rule=lambda m, j: sum(m.x[i, j] for i in m.I) == 1)
    SOLVER.solve(m)
    return m


def ver(m):
    tabla = pd.DataFrame([[pyo.value(m.x[i, j]) + 0.0 for j in m.J] for i in m.I], index=list(m.I), columns=list(m.J))
    print("Costo:", pyo.value(m.obj))
    return tabla


ver(asignar())

Mismo plan que la fuerza bruta: 14. Ahora **relajamos**: x puede valer 0,3 o 0,5.
En la lección 3 eso daba "0,83 de unas baterías". Adivina qué pasa aquí.

In [ ]:
ver(asignar(relajar=True))

**Lectura:** el LP relajado da **exactamente el mismo plan, con 0 y 1**. Nadie pidió que fueran enteros.

¿Casualidad de este ejemplo? Probemos con 300 tablas de costos al azar, de 8 × 8.
Contamos cuántas veces el LP relajado da algún valor fraccional.

In [ ]:
rng = np.random.default_rng(0)
fraccionales = 0
for _ in range(300):
    c = pd.DataFrame(rng.integers(1, 100, (8, 8)), index=[f"i{k}" for k in range(8)], columns=[f"p{k}" for k in range(8)])
    m = asignar(c, relajar=True)
    fraccionales += any(1e-6 < pyo.value(v) < 1 - 1e-6 for v in m.x.values())
print("Tablas con algún x fraccional:", fraccionales, "de 300")

## 4. Por qué sale entero: la matriz totalmente unimodular
Escribamos las restricciones como una tabla: una fila por restricción, una columna por variable.
Cada celda es el número que multiplica a esa variable en esa restricción. Es la **matriz de restricciones**.

In [ ]:
m = asignar()
pares = [(i, j) for i in m.I for j in m.J]
matriz = pd.DataFrame(0, index=[f"ing: {i}" for i in m.I] + [f"proy: {j}" for j in m.J],
                      columns=[f"{i[0]}-{j[:3]}" for i, j in pares])
for k, (i, j) in enumerate(pares):
    matriz.loc[f"ing: {i}", matriz.columns[k]] = 1
    matriz.loc[f"proy: {j}", matriz.columns[k]] = 1
matriz

**Lo que tiene de especial:**
- Todos los números son 0 o 1.
- Cada columna tiene **exactamente dos 1**: uno en la fila de su ingeniero y otro en la fila de su proyecto.
- Las filas se parten en dos grupos (ingenieros y proyectos), y cada columna toca una fila de cada grupo.

Una matriz así es **totalmente unimodular** (*totally unimodular*, TU). La definición formal:
todo determinante de cualquier submatriz cuadrada vale −1, 0 o 1. No hace falta calcularlo; lo que importa es el efecto:

> Si la matriz es TU y los lados derechos son enteros, **todas las esquinas** de la región factible tienen coordenadas enteras.

Y en la lección 1 viste que el LP siempre termina en una esquina. Entonces el LP termina en un plan entero.
El solver resuelve un LP, rápido, y no necesita branch & bound.

**Analogía.** Un parqueadero pintado en el piso. Los carros solo pueden quedar dentro de una raya (las esquinas).
Si todas las rayas están pintadas en números enteros, ningún carro queda "en 0,83 de puesto".

**Otros modelos con matriz TU** (los ves en la lección 5): transporte, flujo en redes, camino más corto.
Por eso esos modelos se resuelven con millones de variables.

## 5. Lo que rompe la suerte
La suerte depende de la forma de la matriz. Una sola restricción con otros números la rompe.

**Ejemplo.** Cada ingeniero tiene un **nivel** de experiencia (1 a 5): Ana 4, Bartek 2, Camilo 3.
El eólico es el proyecto delicado: el cliente exige que su ingeniero tenga **nivel 3 o más**.

El plan de 14 mil € pone a Bartek (nivel 2) en el eólico. Ya no sirve.
La regla se escribe así: Σ_i nivel_i · x[i, eolico] ≥ 3.

In [ ]:
NIVEL = {"Ana": 4, "Bartek": 2, "Camilo": 3}


def asignar_con_nivel(relajar):
    m = pyo.ConcreteModel()
    m.I, m.J = pyo.Set(initialize=list(COSTO.index)), pyo.Set(initialize=list(COSTO.columns))
    m.x = pyo.Var(m.I, m.J, within=pyo.UnitInterval if relajar else pyo.Binary)
    m.obj = pyo.Objective(expr=sum(COSTO.loc[i, j] * m.x[i, j] for i in m.I for j in m.J), sense=pyo.minimize)
    m.ingeniero = pyo.Constraint(m.I, rule=lambda m, i: sum(m.x[i, j] for j in m.J) == 1)
    m.proyecto = pyo.Constraint(m.J, rule=lambda m, j: sum(m.x[i, j] for i in m.I) == 1)
    m.nivel_eolico = pyo.Constraint(expr=sum(NIVEL[i] * m.x[i, "eolico"] for i in m.I) >= 3)
    SOLVER.solve(m)
    return m


print("LP relajado:")
display(ver(asignar_con_nivel(relajar=True)))
print("MILP:")
display(ver(asignar_con_nivel(relajar=False)))

**Lectura:**
- El LP relajado **parte personas**: medio Ana y medio Bartek en el eólico. Nivel 0,5 · 4 + 0,5 · 2 = 3: cumple "en promedio".
  Cuesta 14,5. Es la trampa de la lección 3 otra vez: medio ingeniero no existe.
- El MILP da un plan real: Ana → eólico, Bartek → solar, Camilo → subestación. Cuesta 15.

La nueva fila tiene los números 4, 2, 3: ya no son solo 0 y 1. La matriz deja de ser TU y vuelven las esquinas fraccionales.

**Regla de trabajo:**
- Asignación pura (y sus primos: transporte, redes): resuelve rápido aunque sea enorme.
- Asignación **más reglas extra** con otros coeficientes (nivel, presupuesto, horas): es un MILP de verdad.
  Suele seguir siendo rápido, porque la estructura de asignación ayuda, pero ya hay branch & bound.
- Para saberlo en la práctica: resuelve el LP relajado y cuenta fraccionales. Es lo que hiciste arriba.

## 6. Variantes que conservan la suerte
En la vida real casi nunca hay tantos ingenieros como proyectos. Estas variantes **siguen siendo TU**
(los lados derechos deben ser enteros):

| Situación | Restricción por ingeniero | Restricción por proyecto |
|---|---|---|
| Uno a uno (sección 3) | Σ_j x = 1 | Σ_i x = 1 |
| Sobran ingenieros | Σ_j x **≤** 1 (algunos quedan en la banca) | Σ_i x = 1 |
| Un proyecto necesita varios | Σ_j x ≤ 1 | Σ_i x = **necesita_j** |
| Un ingeniero puede ir a varios | Σ_j x ≤ **capacidad_i** | Σ_i x = necesita_j |

La última fila se llama **problema de transporte**: oferta (ingenieros) y demanda (proyectos). Lo ves en la lección 5.
La sección 7 usa la tercera fila: cada ingeniero va a un solo proyecto, y cada proyecto necesita 1, 2 o 3 ingenieros.

## 7. El modelo crece: 50 ingenieros × 20 proyectos
Los 20 proyectos son los de la cartera de la lección 3. Ahora hay que armar los equipos.

Dos tablas, como en SQL:
1. **Proyectos:** especialidad, región, días de trabajo, cuántos ingenieros necesita, nivel mínimo,
   si exige polaco (trámites con la autoridad local) y si es en el mar (*offshore*: exige certificado).
2. **Ingenieros:** especialidad, nivel, región donde vive, si habla polaco, si tiene certificado offshore y su tarifa por día.

El costo de poner al ingeniero i en el proyecto j:
> costo = días_j × (tarifa_i + 100 € de dieta si trabaja fuera de su región)

In [ ]:
df_proy = pd.read_csv(io.StringIO("""proyecto,especialidad,region,dias,necesita,nivel_min,polaco,offshore
solar_sur,solar,sur,40,3,3,1,0
solar_techos,solar,centro,15,1,1,1,0
solar_norte,solar,norte,35,3,2,1,0
baterias_sur,solar,sur,25,2,3,0,0
baterias_norte,solar,norte,25,2,2,1,0
eolico_costa_a,eolico,costa,60,3,4,0,1
eolico_costa_b,eolico,costa,55,2,3,0,1
eolico_monte,eolico,sur,45,3,3,1,0
repotenciar_eolico,eolico,norte,20,1,2,1,0
subestacion_norte,red,norte,30,2,3,1,0
subestacion_sur,red,sur,30,2,2,1,0
linea_costa,red,costa,50,3,3,0,0
medidores_inteligentes,red,centro,20,1,1,1,0
scada_nuevo,digital,centro,30,2,3,0,0
gemelo_digital,digital,centro,25,2,4,0,0
mantenimiento_predictivo,digital,centro,20,2,2,0,0
biogas_granjas,procesos,sur,25,2,2,1,0
calor_residual,procesos,norte,15,1,1,1,0
bombas_calor,procesos,centro,15,1,1,1,0
hidrogeno_piloto,procesos,costa,35,2,4,0,0
"""))
print("Puestos a cubrir:", df_proy["necesita"].sum())
df_proy.head()

Los ingenieros son simulados con semilla fija, como en la lección 2: siempre salen iguales.

In [ ]:
rng = np.random.default_rng(1)
nombres = ["Ana", "Bartek", "Camilo", "Dorota", "Esteban", "Filip", "Gabriela", "Hanna", "Iván", "Julia",
           "Kacper", "Laura", "Marek", "Natalia", "Óscar", "Paweł", "Renata", "Szymon", "Tomás", "Urszula",
           "Valentina", "Wojtek", "Ximena", "Zofia", "Adam", "Beata", "Carlos", "Daria", "Emil", "Felipe",
           "Grzegorz", "Helena", "Igor", "Jakub", "Karolina", "Lucía", "Michał", "Nicolás", "Ola", "Piotr",
           "Rafał", "Sara", "Tadeusz", "Ula", "Wiktor", "Yolanda", "Zuzanna", "Andrés", "Bogdan", "Kasia"]
cuantos = {"solar": 12, "eolico": 13, "red": 10, "digital": 6, "procesos": 9}
especialidad = np.array([e for e, n in cuantos.items() for _ in range(n)])
rng.shuffle(especialidad)
nivel = rng.integers(1, 6, 50)
nivel[especialidad == "digital"] = [4, 3, 3, 2, 2, 1]  # pocos expertos digitales: es a propósito

df_ing = pd.DataFrame({"ingeniero": nombres, "especialidad": especialidad, "nivel": nivel,
                       "region": rng.choice(["norte", "centro", "sur", "costa"], 50),
                       "polaco": (rng.random(50) < 0.7).astype(int)})
df_ing["offshore"] = ((df_ing["especialidad"] == "eolico") & (rng.random(50) < 0.75)).astype(int)
df_ing["tarifa"] = (1000 * (0.35 + 0.1 * df_ing["nivel"] + rng.uniform(-0.05, 0.05, 50))).round(-1)  # € por día

print(df_ing.shape)
df_ing.head()

### 7.1 Pares factibles: crear solo lo posible
Con 50 ingenieros y 20 proyectos hay 50 × 20 = **1.000 pares**. La mayoría no tiene sentido:
una experta en redes no dirige un parque eólico.

Dos formas de armar el modelo:
- **Mala:** crear las 1.000 variables y agregar restricciones "x = 0" para los pares imposibles.
- **Buena:** crear **solo los pares posibles**. Las variables que no existen no cuestan nada.

El par posible sale de un **join**, como en SQL:
```sql
SELECT ...
FROM proyectos p
JOIN ingenieros i ON i.especialidad = p.especialidad
WHERE i.nivel >= p.nivel_min
  AND (p.polaco = 0 OR i.polaco = 1)
  AND (p.offshore = 0 OR i.offshore = 1)
```
En pandas: `merge` es el `JOIN`, y el filtro con `&` es el `WHERE`.

In [ ]:
pares = df_proy.merge(df_ing, on="especialidad", suffixes=("_p", "_i"))
print("Después del join por especialidad:", len(pares))

posible = ((pares["nivel"] >= pares["nivel_min"])
           & ((pares["polaco_p"] == 0) | (pares["polaco_i"] == 1))
           & ((pares["offshore_p"] == 0) | (pares["offshore_i"] == 1)))
pares = pares[posible].copy()
pares["fuera_de_casa"] = (pares["region_p"] != pares["region_i"]).astype(int)
pares["costo"] = pares["dias"] * (pares["tarifa"] + 100 * pares["fuera_de_casa"])

print("Después del filtro:", len(pares), "pares factibles de", len(df_ing) * len(df_proy))
print("Par más caro:", eur(pares["costo"].max()))
pares[["ingeniero", "proyecto", "nivel", "nivel_min", "fuera_de_casa", "costo"]].head()

**Lectura:**
- El join por especialidad deja 206 pares. El filtro deja **140 de 1.000**: 86 % menos variables.
- El par más caro cuesta 55.200 €: 60 días de eólico en el mar, con dieta. Guarda ese número para la sección 7.3.

Tu modelo de clientes y asesores (`python/assign_clients_workers.py`) hace lo mismo:
600 clientes × 40 asesores serían 24.000 variables, y el join deja solo los pares de la misma oficina y el cargo correcto.
Con modelos de verdad, es la diferencia entre resolver en segundos y no poder ni armar el modelo.

### 7.2 Revisar antes de resolver
Antes de cocinar, revisas que estén todos los ingredientes. Antes de resolver, revisas que cada proyecto
tenga **suficientes candidatos**. Si no, el modelo es infactible y el solver solo dice "infeasible", sin decir por qué.

In [ ]:
revision = (pares.groupby("proyecto").size().rename("candidatos").to_frame()
            .join(df_proy.set_index("proyecto")["necesita"], how="right")
            .fillna(0).astype(int))
revision["faltan"] = (revision["necesita"] - revision["candidatos"]).clip(lower=0)
revision.sort_values("candidatos").head(6)

**Lectura:** `gemelo_digital` necesita 2 ingenieros digitales de nivel 4 o más. Solo hay **1** candidato.
Ese modelo, tal como está, es **infactible**. Ningún solver lo arregla.

Esta revisión no garantiza que todo lo demás quepa: dos proyectos pueden pelear por los mismos candidatos.
Pero atrapa el 80 % de los errores de datos en una línea. Hazla siempre.

### 7.3 El modelo elástico: que no se caiga si falta gente
En vez de dejar que el modelo sea infactible, le damos una salida cara:
- **u_j** = puestos del proyecto j que quedan **sin cubrir**.
- Cada puesto sin cubrir cuesta **100.000 €**: contratar un consultor externo.

> Σ_i x[i, j] + u_j = necesita_j

Si falta gente, el solver usa u_j. Si hay gente, no lo usa, porque es más caro que cualquier ingeniero.
Es la misma idea de `u[c]` en tu modelo de clientes y asesores.

**¿Por qué 100.000 y no 50.000?** El par más caro cuesta 55.200 €. Con un externo a 50.000 €,
al solver le sale más barato el externo que un ingeniero propio en `eolico_costa_a`, y deja 3 puestos sin cubrir
aunque haya gente. La penalización tiene que ser **mayor que el par más caro**. Es la regla del big-M de la lección 3,
con otra cara: ni tan chica que cambie la respuesta, ni absurda (10⁹) por si acaso.

**Detalle de código.** Las restricciones se arman desde los pares, con `groupby`.
Así cada restricción solo suma las variables que existen. Un ingeniero sin ningún par posible no tiene restricción
(si la creas, Pyomo da error: "la restricción es trivialmente verdadera").

In [ ]:
COSTO_EXTERNO = 100_000


def equipos(pares, df_proy, relajar=False):
    A = list(zip(pares["ingeniero"], pares["proyecto"]))
    costo = dict(zip(A, pares["costo"]))
    por_ing = pares.groupby("ingeniero")["proyecto"].apply(list).to_dict()
    por_proy = pares.groupby("proyecto")["ingeniero"].apply(list).to_dict()
    necesita = df_proy.set_index("proyecto")["necesita"].to_dict()

    m = pyo.ConcreteModel()
    m.A = pyo.Set(initialize=A, dimen=2)
    m.I = pyo.Set(initialize=list(por_ing))
    m.J = pyo.Set(initialize=list(necesita))
    m.x = pyo.Var(m.A, within=pyo.UnitInterval if relajar else pyo.Binary)
    m.u = pyo.Var(m.J, within=pyo.NonNegativeReals)  # puestos sin cubrir
    m.obj = pyo.Objective(expr=sum(costo[a] * m.x[a] for a in m.A) + COSTO_EXTERNO * sum(m.u[j] for j in m.J),
                          sense=pyo.minimize)
    m.ingeniero = pyo.Constraint(m.I, rule=lambda m, i: sum(m.x[i, j] for j in por_ing[i]) <= 1)
    m.proyecto = pyo.Constraint(m.J, rule=lambda m, j: sum(m.x[i, j] for i in por_proy.get(j, [])) + m.u[j] == necesita[j])
    if relajar:  # los precios sombra solo existen en el LP
        m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    SOLVER.solve(m)
    return m


inicio = time.perf_counter()
me = equipos(pares, df_proy)
print(f"Costo total: {eur(pyo.value(me.obj))} | resuelto en {time.perf_counter() - inicio:.2f} s")
print("Puestos sin cubrir:", {j: pyo.value(me.u[j]) for j in me.J if pyo.value(me.u[j]) > 0.5})

Comprobemos la lección de la sección 4 con el modelo grande: ¿el LP relajado sale entero?

In [ ]:
ml = equipos(pares, df_proy, relajar=True)
print("LP relajado:", eur(pyo.value(ml.obj)), "| MILP:", eur(pyo.value(me.obj)))
print("x fraccionales en el LP:", sum(1e-6 < pyo.value(v) < 1 - 1e-6 for v in ml.x.values()))

**Lectura:** mismo costo y cero fraccionales. La columna de u_j tiene un solo 1 (solo aparece en la fila de su proyecto),
y eso no rompe la matriz TU. El modelo de 50 × 20 es, en el fondo, un LP.

### 7.4 La solución, de vuelta a tablas

In [ ]:
asignados = pares[[pyo.value(me.x[a]) > 0.5 for a in zip(pares["ingeniero"], pares["proyecto"])]]
res_proy = (asignados.groupby("proyecto")
            .agg(equipo=("ingeniero", ", ".join), costo=("costo", "sum"), fuera_de_casa=("fuera_de_casa", "sum"))
            .reindex(df_proy["proyecto"]))
res_proy["sin_cubrir"] = [round(pyo.value(me.u[j])) for j in res_proy.index]
banca = df_ing[~df_ing["ingeniero"].isin(asignados["ingeniero"])]

print("Ingenieros asignados:", len(asignados), "| en la banca:", len(banca))
print("Costo de ingenieros:", eur(asignados["costo"].sum()), "| dietas:", eur(100 * (asignados["dias"] * asignados["fuera_de_casa"]).sum()))

**¿Dónde se va la plata?** Los 5 proyectos más caros.

In [ ]:
res_proy.nlargest(5, "costo")

**¿Quién quedó en la banca?** Están libres para otro proyecto o para capacitación.

In [ ]:
banca[["ingeniero", "especialidad", "nivel", "region", "polaco", "offshore"]]

**¿Quiénes trabajan lejos de casa?** Cada uno paga 100 € de dieta por día.

In [ ]:
asignados[asignados["fuera_de_casa"] == 1][["ingeniero", "region_i", "proyecto", "region_p", "dias"]].sort_values("dias", ascending=False).head()

**Lectura para el jefe de recursos humanos:**
- 39 de 40 puestos cubiertos. Falta 1 en `gemelo_digital`: hay que contratar o formar a alguien digital de nivel 4.
- El proyecto más caro es `eolico_costa_a`: 151.800 €. Sus 3 ingenieros viajan desde el centro.
- En la banca hay 11 personas, casi todas de la **costa**. Pero ninguna sirve para `eolico_costa_a`:
  ahí se pide nivel 4 y certificado offshore, y los de la costa son de nivel bajo o no tienen certificado.
- Formar a 3 ingenieros de la costa con nivel y certificado ahorraría 3 × 60 días × 100 € = 18.000 € de dietas en ese proyecto.

## 8. Los precios sombra vuelven a valer
En la lección 3 dijiste adiós a los precios sombra: en un MILP la ganancia cambia a saltos.
Pero aquí el MILP **es** un LP disfrazado (la matriz es TU). Entonces los precios sombra de la lección 2 vuelven a tener sentido.

Dos preguntas para el jefe de recursos humanos:
- **Por proyecto:** ¿cuánto cuesta **un puesto más** en ese proyecto? Es el precio sombra de su restricción.
- **Por ingeniero:** ¿cuánto **ahorra** tener a esta persona? Es el precio sombra de su restricción "≤ 1".
  Sale negativo o 0: tener a alguien disponible solo puede bajar el costo.

Pyomo lee los precios sombra del LP relajado (`ml`), que es el mismo óptimo.

In [ ]:
precio_proy = pd.Series({j: ml.dual[ml.proyecto[j]] for j in ml.J}, name="costo_puesto_extra").round(0)
precio_ing = pd.Series({i: -ml.dual[ml.ingeniero[i]] for i in ml.I}, name="ahorro_que_aporta").round(0) + 0.0

print("Puestos más caros de cubrir:")
display(precio_proy.sort_values(ascending=False).head(5).to_frame())
print("Ingenieros que más ahorro aportan:")
display(precio_ing.sort_values(ascending=False).head(5).to_frame())

**Cómo se lee:**
- `gemelo_digital`, `scada_nuevo` e `hidrogeno_piloto`: un puesto más cuesta **100.000 €**, un externo.
  En los tres ya no queda nadie libre que califique. El de `scada_nuevo` es menos obvio: tiene 3 candidatos,
  pero Camilo está en el gemelo digital.
- `eolico_costa_a` y `eolico_costa_b`: un puesto más cuesta unos 55.000 €. Hay gente, pero es cara y viene de lejos.
- **Camilo** aporta 81.750 €: es el único ingeniero digital de nivel 4. Sin él, el gemelo digital necesita 2 externos.
  Ese es el número para decidir **a quién retener**, o a quién no mandar de vacaciones en plena obra.
- En el top 5 de ingenieros solo hay perfiles digitales y de procesos: son las especialidades con menos gente.

**Advertencia de la lección 2.** En asignación los precios sombra a veces **no son únicos**
(varios juegos de precios dan el mismo óptimo). Indican dónde mirar. El re-solve dice cuánto.

Comprobemos con Camilo: lo sacamos de los pares y re-resolvemos.

In [ ]:
clave = precio_ing.idxmax()
sin_clave = equipos(pares[pares["ingeniero"] != clave], df_proy)
print(f"Sin {clave}: {eur(pyo.value(sin_clave.obj))} (sube {eur(pyo.value(sin_clave.obj) - pyo.value(me.obj))})")
print(f"Predicho por el precio sombra: {eur(precio_ing[clave])}")

**Lectura:** el re-solve da exactamente lo que dijo el precio sombra: 81.750 €.
Con la matriz TU, la lección 2 vuelve a funcionar.

## 9. Un algoritmo especial: el método húngaro
La asignación pura es tan común que tiene su propio algoritmo, el **método húngaro** (Kuhn, 1955).
En Python está en `scipy.optimize.linear_sum_assignment`. No necesita Pyomo ni solver.

Para usarlo con el modelo de 50 × 20 hay que darle una **matriz cuadrada o rectangular**:
- Cada proyecto se parte en **puestos**: `solar_sur` necesita 3 → 3 columnas.
- Los pares imposibles cuestan 100.000 €, igual que un externo. Si el método los elige, es un puesto sin cubrir.

In [ ]:
puestos = [(j, k) for j, n in df_proy.set_index("proyecto")["necesita"].items() for k in range(n)]
costo_par = pares.set_index(["ingeniero", "proyecto"])["costo"].to_dict()
C = np.array([[costo_par.get((i, j), COSTO_EXTERNO) for j, _ in puestos] for i in df_ing["ingeniero"]])
print("Matriz:", C.shape, "(ingenieros × puestos)")

inicio = time.perf_counter()
filas_i, cols_j = linear_sum_assignment(C)
print(f"Método húngaro: {eur(C[filas_i, cols_j].sum())} | {1000 * (time.perf_counter() - inicio):.1f} ms")
print(f"Pyomo + HiGHS:  {eur(pyo.value(me.obj))}")

**Lectura:** el mismo costo, en menos de 1 milisegundo.

**¿Cuándo usar cada uno?**
- **Método húngaro:** asignación pura, sin reglas extra. Muy rápido, pero no acepta ni una restricción más.
- **MILP (Pyomo):** en cuanto aparece una regla de negocio (nivel del equipo, presupuesto, "Ana y Bartek no juntos"),
  como en la sección 5. En la industria casi siempre aparece.

## 10. Resumen: conceptos de hoy
| Concepto | Qué es | En la lección | En inglés |
|---|---|---|---|
| Asignación | Cada persona a una tarea, cada tarea a una persona, al menor costo | 3 ingenieros, 3 proyectos | assignment problem |
| Goloso | Tomar en cada paso lo mejor que se ve | 15 contra 14 del óptimo | greedy |
| Matriz de restricciones | Coeficientes de cada variable en cada restricción | dos 1 por columna | constraint matrix |
| Totalmente unimodular | Toda submatriz cuadrada tiene determinante −1, 0 o 1: esquinas enteras | el LP sale entero | totally unimodular (TU) |
| Integralidad | Si la matriz es TU y los lados derechos enteros, el LP da enteros | 300 de 300 sin fraccionales | integrality property |
| Pares factibles | Crear solo las variables posibles | 140 de 1.000 | sparse index set |
| Modelo elástico | Variable de holgura cara para que el modelo no sea infactible | consultor externo a 100.000 € | elastic model, slack penalty |
| Precio sombra en asignación | Con matriz TU vuelve a valer | Camilo aporta 81.750 € | dual value |
| Transporte | Asignación con oferta y demanda mayores que 1 | lección 5 | transportation problem |
| Método húngaro | Algoritmo exacto para asignación pura | `linear_sum_assignment` | Hungarian algorithm |

## 11. Ejercicios
1. **Contratar al experto.** RRHH puede contratar a una ingeniera digital de nivel 5, que vive en el centro, habla polaco
   y cobra 900 € por día. Antes de resolver: según el precio sombra de `gemelo_digital`, ¿cuánto debería bajar el costo total?
   Agrégala a `df_ing`, rehaz los pares y comprueba.
2. **Todo en polaco.** La autoridad pide que **todos** los proyectos tengan ingenieros que hablen polaco.
   Cambia la columna `polaco` de `df_proy` a 1 en todas las filas. ¿Cuántos puestos quedan sin cubrir? ¿En qué proyectos?
   Primero revisa con la tabla de la sección 7.2.
3. **Trabajar cerca de casa.** Prueba la regla "nadie trabaja fuera de su región" como un filtro más en el join.
   ¿Cuántos puestos quedan sin cubrir y cuánto cuesta? ¿Vale la pena la regla?

In [ ]:
# Escribe aquí tus intentos

## En la entrevista
Pregunta típica: *"Your assignment model has binary variables. Why does it solve so fast?"*
> *"Because the constraint matrix of a pure assignment or transportation problem is totally unimodular.
> With integer right-hand sides, every vertex of the LP is integral, so the LP relaxation already returns 0/1 and
> no branching is needed. That breaks as soon as I add a side constraint with general coefficients,
> like a minimum team skill level or a budget. Then it is a real MILP, though the assignment structure still helps."*

Seguimiento: *"How do you build it at scale?"*
> *"I only create variables for feasible pairs, with a join on skills and filters on level, language or certificates.
> Before solving I check that each task has enough candidates. And I add a penalized slack per task, so the model
> never comes back infeasible: it tells me where I'm short of people instead."*

## Soluciones (no mires antes de intentar)

In [ ]:
# Ejercicio 1
print("Predicho por el precio sombra: baja hasta", eur(precio_proy["gemelo_digital"]), "menos el costo de la nueva")
nueva = pd.DataFrame([{"ingeniero": "Nueva", "especialidad": "digital", "nivel": 5, "region": "centro",
                       "polaco": 1, "offshore": 0, "tarifa": 900}])


def armar_pares(df_proy, df_ing, solo_su_region=False):
    p = df_proy.merge(df_ing, on="especialidad", suffixes=("_p", "_i"))
    ok = ((p["nivel"] >= p["nivel_min"]) & ((p["polaco_p"] == 0) | (p["polaco_i"] == 1))
          & ((p["offshore_p"] == 0) | (p["offshore_i"] == 1)))
    if solo_su_region:
        ok &= p["region_p"] == p["region_i"]
    p = p[ok].copy()
    p["fuera_de_casa"] = (p["region_p"] != p["region_i"]).astype(int)
    p["costo"] = p["dias"] * (p["tarifa"] + 100 * p["fuera_de_casa"])
    return p


m1 = equipos(armar_pares(df_proy, pd.concat([df_ing, nueva], ignore_index=True)), df_proy)
print("Costo con la nueva:", eur(pyo.value(m1.obj)), "| baja", eur(pyo.value(me.obj) - pyo.value(m1.obj)))

El precio sombra del gemelo digital es 100.000 €: la nueva reemplaza a un externo. Ella cuesta 25 días × 900 € = 22.500 €.
Predicción: baja 100.000 − 22.500 = **77.500 €**. El re-solve da exactamente eso.
Contratarla para un proyecto de 25 días cuesta menos que un externo: conviene.

In [ ]:
# Ejercicio 2
todo_polaco = df_proy.assign(polaco=1)
m2 = equipos(armar_pares(todo_polaco, df_ing), todo_polaco)
print("Costo:", eur(pyo.value(m2.obj)))
print("Sin cubrir:", {j: round(pyo.value(m2.u[j])) for j in m2.J if pyo.value(m2.u[j]) > 0.5})

Quedan **6 puestos sin cubrir**: los 3 de `eolico_costa_a` (Dorota y Renata no hablan polaco, y los offshore que quedan
no alcanzan para los tres eólicos que ahora exigen polaco), 1 en `solar_sur`, 1 en `scada_nuevo` y el de siempre en `gemelo_digital`. El costo sube 292.900 €.
Ese es el número para negociar con la autoridad, o el presupuesto de un curso de polaco.

In [ ]:
# Ejercicio 3
m3 = equipos(armar_pares(df_proy, df_ing, solo_su_region=True), df_proy)
print("Costo:", eur(pyo.value(m3.obj)), "| sube", eur(pyo.value(m3.obj) - pyo.value(me.obj)))
print("Sin cubrir:", sum(round(pyo.value(m3.u[j])) for j in m3.J), "puestos")

**24 puestos sin cubrir** y el costo sube 1,58 M€. Las dietas de hoy suman solo 92.000 €.
La regla ahorra como mucho 92.000 € y cuesta 1,58 M€: no vale la pena.
Un filtro en el join parece inofensivo, pero cambia todo el modelo. Revisa siempre con la tabla de la sección 7.2.